In [ ]:
!nc -vz pub.worldb.dedyn.io 9003

# Databricks Free Edition doesn't include the Trino JDBC driver

In [ ]:
trino_host = "pub.worldb.dedyn.io"
trino_port = 9003

trino_user = "my-user"
trino_password = "my-password"

catalog = "postgresql"
schema = "public"

jdbc_url = (
    f"jdbc:trino://{trino_host}:{trino_port}/{catalog}/{schema}"
    "?SSL=true"
    "&SSLVerification=FULL"
)

df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("driver", "io.trino.jdbc.TrinoDriver")
    .option("user", trino_user)
    .option("password", trino_password)
    .option("query", "SELECT * FROM my_table LIMIT 10")
    .load()
)

#display(df)

# Workaround: use the Trino Python client

In [ ]:
%pip install trino cryptography

In [ ]:
!ls -la /Volumes/workspace/default/on_premises_certificates/trino

In [ ]:
!ls -la /Volumes/workspace/default/on_premises_certificates/trino/ca.crt
!ls -la /Volumes/workspace/default/on_premises_certificates/trino/trino-client.p12

In [ ]:
for scope in dbutils.secrets.listScopes():
    print(f"\nScope: {scope.name}")
    for secret in dbutils.secrets.list(scope.name):
        print(f"  {secret.key}")

dbutils.secrets.list("on-premises-integration")    

In [ ]:
import os
import tempfile
from pathlib import Path

from cryptography.hazmat.primitives.serialization import (
    Encoding,
    NoEncryption,
    PrivateFormat,
    pkcs12,
)

def prepare_trino_mtls(client_keystore_path, client_keystore_password):
    private_key, client_certificate, _ = pkcs12.load_key_and_certificates(
        client_keystore_path.read_bytes(),
        client_keystore_password,
    )

    if private_key is None or client_certificate is None:
        raise ValueError(
            "trino-client.p12 does not contain a client identity"
        )

    temporary_dir = tempfile.TemporaryDirectory(prefix="trino-mtls-")
    temporary_path = Path(temporary_dir.name)

    client_certificate_path = temporary_path / "client.crt"
    client_key_path = temporary_path / "client.key"

    client_certificate_path.write_bytes(
        client_certificate.public_bytes(Encoding.PEM)
    )

    client_key_path.write_bytes(
        private_key.private_bytes(
            Encoding.PEM,
            PrivateFormat.PKCS8,
            NoEncryption(),
        )
    )

    os.chmod(client_key_path, 0o600)

    return (
        temporary_dir,
        client_certificate_path,
        client_key_path,
    )


ca_path = Path(
    "/Volumes/workspace/default/on_premises_certificates/trino/ca.crt"
)

client_keystore_path = Path(
    "/Volumes/workspace/default/on_premises_certificates/trino/trino-client.p12"
)

client_keystore_password = dbutils.secrets.get(
    scope="on-premises-integration",
    key="trino-client-keystore-password",
).encode()


temporary_dir, client_certificate_path, client_key_path = prepare_trino_mtls(
    client_keystore_path,
    client_keystore_password,
)

In [ ]:
!nc -vz pub.worldb.dedyn.io 9003

In [ ]:
temporary_dir, client_certificate_path, client_key_path = ( prepare_trino_mtls( client_keystore_path, client_keystore_password, ) )


In [ ]:

import trino
from trino.auth import CertificateAuthentication

conn = trino.dbapi.connect(
    host="pub.worldb.dedyn.io",
    port=9003,
    user="trino-client",
    catalog="postgresql",
    schema="public",
    http_scheme="https",
    auth=CertificateAuthentication(
        str(client_certificate_path),
        str(client_key_path),
    ),
    verify=str(ca_path),
)

In [ ]:
cursor = conn.cursor()

In [ ]:
cursor.execute("SELECT version()")
rows = cursor.fetchall()
display(rows)

In [ ]:
cursor.execute("SELECT * FROM system.runtime.queries")
rows = cursor.fetchall()
display(rows)

In [ ]:
conn.close()
temporary_dir.cleanup()